# Week 8 - Advanced Classification & Clustering

## Machine Learning for Engineering Applications

## By the end of this seminar you should be able to:
1. Tune and optimise classification models using:

    - Cross-validation
    - Grid Search
    - Ensemble methods

2. Interpret model decisions using:

    - Feature importance
    - SHAP values
    - Probability calibration

3. Discover structure in unlabeled engineering datasets using:

    - k-Means
    - Hierarchical Clustering
    - DBSCAN

4. Compare clustering algorithms and understand their strengths and weaknesses.

5. Apply practical Python workflows using:

    - scikit-learn
    - XGBoost
    - SHAP
    - seaborn
    - matplotlib

## Dataset Overview: Predictive Maintenance of Rotating Machinery
Predictive maintenance aims to identify developing faults in machines before catastrophic failure occurs. Modern condition-monitoring systems collect sensor measurements such as vibration amplitude, temperature and rotational speed, which can be analysed using machine learning algorithms to classify machine health states.

In this lab, you will work with a vibration monitoring dataset collected from a simulated industrial rotating machine. This the same dataset which we looked at in week 7 seminar. The objective is to use machine learning classification techniques to identify the health condition of the machine based on measurements obtained from vibration sensors and operating parameters.

The dataset contains observations from four different operating conditions:

* Healthy: Normal machine operation with low vibration levels.
* Run-Up (Transient): Machine accelerating from low to rated speed. These data points represent normal but non-steady-state behaviour.
* Bearing Fault: Simulated inner-race bearing damage, resulting in increased vibration levels and impulsive behaviour.
* Rotor Imbalance: Uneven mass distribution on the rotating shaft, causing elevated vibration at the rotational frequency.

For each observation, several sensor-derived features are available:

* RPM: Rotational speed of the machine.
* Temperature: Operating temperature.
* RMS Vibration: Overall vibration energy.
* Peak Vibration: Maximum vibration amplitude.
* Kurtosis: Indicator of impulsive events often associated with bearing faults.
* Skewness: Measure of the asymmetry of the vibration signal distribution.

Before building a machine learning classification model, you should investigate and address potential data issues through an appropriate preprocessing pipeline which we discussed during the Week 2 lecture on datasets and pre-processing. Typical preprocessing steps include duplicate removal, missing value treatment, outlier analysis, feature scaling, and exploratory data analysis.

The primary task is to use the sensor measurements to **classify** the machine condition (FaultLabel), allowing the development of a simple predictive maintenance model capable of distinguishing between healthy operation and different fault types. This exercise demonstrates the complete workflow from raw engineering data to machine learning-based condition monitoring.


## Part A. Advanced Classification

### Import Libraries

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

### Load Dataset

In [2]:
df = pd.read_csv(
    "data/dataset-classification.csv"
)

df.head()

,RecordID,RPM,Temperature_C,RMS_g,Peak_g,Kurtosis,Skewness,OperatingState,FaultLabel
0,1,1808.941,45.751,0.239,0.838,4.066,-0.082,Healthy,Healthy
1,2,1795.786,48.843,0.243,0.851,2.671,0.190,Healthy,Healthy
2,3,1791.658,45.162,0.227,0.795,1.661,-0.604,Healthy,Healthy
3,4,1789.879,44.177,0.229,0.803,2.364,-0.494,Healthy,Healthy
4,5,1826.382,45.594,0.222,0.777,2.003,-0.191,Healthy,Healthy


### Define Features and Labels

In [3]:
X = df[['RPM',
        'Temperature_C',
        'RMS_g',
        'Peak_g',
        'Kurtosis',
        'Skewness']]

y = df['FaultLabel']

### Train/Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Cross-Validation

#### Random Forest Example

In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42
)

scores = cross_val_score(
    rf,
    X_train,
    y_train,
    cv=5,
    scoring='f1_macro'
)

print(scores)
print(scores.mean())

[0.98643808 0.98958504 0.98437439 0.99062463 0.98958504]
0.988121435090599


### Discussion:

Why is cross-validation better than a single train/test split?

What does the standard deviation tell us?

### Hyperparameter Tuning Using Grid Search

#### Random Forest

In [6]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10]
}

from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1_macro'
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

KeyboardInterrupt: 

### XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_train_enc = encoder.fit_transform(y_train)
y_test_enc = encoder.transform(y_test)

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05
)

xgb.fit(X_train, y_train_enc)

pred = xgb.predict(X_test)

### Compare Multiple Models

In [ ]:
models = {
    "Random Forest": rf,
    "XGBoost": xgb
}

#### You can extend this section using:

    - Logistic Regression
    - kNN
    - SVM

from the previous seminar.

## Part B. Model Interpretation

### Random Forest Feature Importance

In [ ]:
rf.fit(X_train, y_train)

importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
)

importance.sort_values().plot(
    kind='barh'
)

plt.title("Feature Importance")
plt.show()

#### Discussion:

Which sensor measurements matter most?

Are the results physically sensible?

### SHAP Explanations

#### Install:
pip install shap

#### Import:

In [ ]:
import shap

#### Create explainer:

In [ ]:
explainer = shap.TreeExplainer(rf)

shap_values = explainer(X_test)

#### Global explanation:

In [ ]:
shap.summary_plot(
    shap_values,
    X_test
)

#### Interpretation:
A SHAP value indicates how much each feature contributes to moving a prediction away from the baseline prediction made by the model.

Positive SHAP values push the prediction towards a class, while negative SHAP values push the prediction away from that class.

- Which features drive predictions?
- Are effects positive or negative?

### Local SHAP Explanation

In [ ]:
# Explain a single prediction

sample_id = 10

predicted_class = rf.predict(
    X_test.iloc[[sample_id]]
)[0]

class_idx = np.where(
    rf.classes_ == predicted_class
)[0][0]

print(
    f"Predicted class: {predicted_class}"
)

shap.plots.waterfall(
    shap_values[
        sample_id,
        :,
        class_idx
    ]
)

#### Discussion:
The waterfall plot explains how each feature contributes to the model's prediction for a single machine. Features pushing the prediction towards the predicted class appear in red, while features pushing the prediction away from that class appear in blue. This allows engineers to understand which sensor measurements were most influential in the decision-making process.

- Why did the model classify this particular machine as faulty?

## Calibration

### Why Calibration Matters
A model may report:
    95% confidence
    
while only being correct:
    80% of the time
    

### Reliability Curve

In [ ]:
from sklearn.calibration import (
    calibration_curve,
    CalibratedClassifierCV
)

cal_rf = CalibratedClassifierCV(
    rf,
    method='isotonic',
    cv=5
)

cal_rf.fit(X_train, y_train)

from sklearn.calibration import calibration_curve

# Example for one class
target_class = 'InnerRace'

y_true = (y_test == target_class).astype(int)

y_prob = cal_rf.predict_proba(X_test)[
    :, list(cal_rf.classes_).index(target_class)
]

prob_true, prob_pred = calibration_curve(
    y_true,
    y_prob,
    n_bins=10
)

plt.figure(figsize=(6,6))

plt.plot(
    prob_pred,
    prob_true,
    marker='o',
    label='Calibrated Model'
)

plt.plot(
    [0,1],
    [0,1],
    '--',
    color='black',
    label='Perfect Calibration'
)

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Frequency")

plt.title("Reliability Curve")

plt.legend()

plt.show()

Calibration does not necessarily improve classification accuracy. Instead, it improves the quality of the model's probability estimates, which is important when prediction confidence is used for engineering decision-making and maintenance scheduling.

Points lie on y = x: Perfectly calibrated

Points below diagonal: Over-confident model (The model is too confident)

Points above diagonal: Under-confident model (The model is more accurate than it believes)

## Clustering

### Engineering Clustering Dataset

In many engineering applications, labelled data are scarce because faults occur infrequently and manual labelling requires expert knowledge. As a result, unsupervised learning techniques are often used to discover patterns, identify abnormal operating conditions, and support condition monitoring.

The **synthetic** dataset used in this exercise contains measurements that could be obtained from a rotating machine monitoring system:

- RPM
- Temperature
- RMS vibration
- Peak vibration
- Kurtosis
- Skewness

Unlike the classification dataset used previously, this dataset does not contain machine condition labels. Instead, the objective is to determine whether meaningful operating states or fault patterns can be discovered directly from the measurements.

The **synthetic** dataset was designed to contain:

- Healthy operating behaviour
- Early-stage fault behaviour
- Severe fault behaviour
- Early imbalance behaviour
- Severe imbalance behaviour
- A small number of anomalous observations

In this exercise, you will:

1. Visualise the data using dimensionality reduction.
2. Apply k-Means clustering.
3. Apply Hierarchical Clustering.
4. Apply DBSCAN.
5. Compare the strengths and weaknesses of each method.
6. Investigate how different clustering algorithms handle overlapping groups and noisy observations.


### Load Unlabelled Dataset

In [ ]:
df = pd.read_csv(
    "data/dataset-clustering.csv",
)

df.head()

#### Features:

In [ ]:
X = df.drop(columns=['RecordID'])

#### Scale:

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

### PCA Visualisation

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(
    n_components=2
)

X_pca = pca.fit_transform(X_scaled)

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    alpha=0.5
)

plt.title("PCA Projection")
plt.show()

#### Question:
- Can you visually identify clusters?

### k-Means Clustering

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=5,
    random_state=42
)

clusters = kmeans.fit_predict(
    X_scaled
)

#### Visualise:

In [ ]:
plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=clusters,
    cmap='tab10'
)

plt.title("k-Means")
plt.show()

### Selecting k Using Elbow Method

In [ ]:
inertia = []

for k in range(1,11):

    model = KMeans(
        n_clusters=k,
        random_state=42
    )

    model.fit(X_scaled)

    inertia.append(model.inertia_)

plt.plot(
    range(1,11),
    inertia,
    marker='o'
)

plt.xlabel("k")
plt.ylabel("Inertia")
plt.show()

### Hierarchical Clustering

In [ ]:
from scipy.cluster.hierarchy import (
    linkage,
    dendrogram
)

Z = linkage(
    X_scaled,
    method='ward'
)

plt.figure(figsize=(12,5))

dendrogram(Z)

plt.show()

### Agglomerative Clustering

In [ ]:
from sklearn.cluster import AgglomerativeClustering

hc = AgglomerativeClustering(
    n_clusters=5
)

hc_labels = hc.fit_predict(
    X_scaled
)

### PCA Visualisation

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=hc_labels,
    cmap='tab10',
    alpha=0.7
)

plt.xlabel('PC1')
plt.ylabel('PC2')

plt.title('Agglomerative Clustering')

plt.show()

### Nonlinear Visualisation using t-SNE

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(
    n_components=2,
    random_state=42
)

X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(8,6))

plt.scatter(
    X_tsne[:,0],
    X_tsne[:,1],
    alpha=0.6
)

plt.title('t-SNE Projection')

plt.show()

### DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

# Note DBSCAN is highly sensitive to eps and min_samples
db = DBSCAN(
    eps=1.2,
    min_samples=10
)

db_labels = db.fit_predict(
    X_scaled
)

print(pd.Series(db_labels).value_counts())


#### Visualisation:

In [ ]:
plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=db_labels,
    cmap='tab10'
)

plt.title("DBSCAN")
plt.show()

#### Discussion:

- Which points are labelled -1?
- Why are they considered noise?

### Compare Clustering Methods

#### Calculate silhouette score:

In [ ]:
from sklearn.metrics import silhouette_score

#### k-Means

In [ ]:
kmeans_score = silhouette_score(
    X_scaled,
    clusters
)

print(
    f'k-Means Silhouette Score: {kmeans_score:.3f}'
)

#### Hierarchical

In [ ]:
hc_score = silhouette_score(
    X_scaled,
    hc_labels
)

print(
    f'Hierarchical Clustering Silhouette Score: {hc_score:.3f}'
)

##### Display results:

In [ ]:
results = pd.DataFrame({

    'Method': [
        'k-Means',
        'Hierarchical'
    ],

    'Silhouette Score': [
        kmeans_score,
        hc_score
    ]
})

results

##### Visual comparison:

In [ ]:
sns.barplot(
    data=results,
    x='Method',
    y='Silhouette Score'
)

plt.title('Clustering Performance Comparison')

plt.show()

## Final Reflection Questions
1. Did Grid Search improve model performance?
2. Which algorithm performed best after tuning?
3. Which sensor measurements were most important?
4. What additional insight did SHAP provide?
5. Why might a highly accurate model still be poorly calibrated?
6. How many clusters were suggested by the Elbow Method?
7. Which clustering algorithm best handled anomaly points?
8. Why might DBSCAN outperform k-Means in noisy engineering datasets?
9. If this dataset came from a real machine, what additional sensors would you like to have available?
